In [5]:
import tdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyranges as pr
from sklearn.decomposition import PCA
import seaborn as sb
from scipy.stats import pearsonr

# custom
# from QC_utils import * # my QC functions

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
from analysis_utils import *

# Add labels for TEs, genomic regions, and disease association

This data was generated by get_flanking_repeats_all_families.sh

In [6]:
# read the summary
revised_labels= pd.read_csv("flanking_repeats_summary.tsv",sep="\t",header=0)
revised_labels

,STR_chr,STR_start,STR_end,STR_ID,upstream_coords,upstream_family,upstream_ID,upstream_distance,downstream_coords,downstream_family,downstream_ID,downstream_distance,STR_classification
0,chr1,14069,14081,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
1,chr1,15811,15822,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
2,chr1,16619,16632,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
3,chr1,16716,16727,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
4,chr1,19175,19184,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2430070,chrY,56694118,56694136,4542235.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
2430071,chrY,56694221,56694236,4542236.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
2430072,chrY,56694333,56694350,4542237.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none
2430073,chrY,56694379,56694395,4542238.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,none


In [7]:
# fill STR_classification none with non_mei
revised_labels.loc[revised_labels["STR_classification"] == "none", "STR_classification"] = "non_mei"
revised_labels["internal_external"] = np.where((revised_labels["STR_classification"].str.contains("internal")), "internal", "external")
revised_labels["internal_external"] = np.where(revised_labels["STR_classification"] == "non_mei", "non_mei", revised_labels["internal_external"])
revised_labels.internal_external.value_counts()


internal_external
non_mei     1191459
internal     894306
external     344310
Name: count, dtype: int64

In [8]:
updated_labels = pd.read_csv("../QC_scripts/final_updated_labels.tsv", sep="\t")
updated_labels

,LocusID,original_LocusID,TRID,chrom,start,end,str_period,str_motif,satellite_allele_labelled,exclusion_type
0,15.0,15.0,1-14069-14081-CCTC,chr1,14069,14081,4.0,CCTC,False,keep
1,16.0,16.0,1-15811-15822-CTG,chr1,15811,15822,3.0,CTG,False,keep
2,17.0,17.0,1-16619-16632-GCT,chr1,16619,16632,3.0,GCT,False,keep
3,18.0,18.0,1-16716-16727-GGT,chr1,16716,16727,3.0,GGT,False,keep
4,19.0,19.0,1-19175-19184-TCC,chr1,19175,19184,3.0,TCC,False,keep
...,...,...,...,...,...,...,...,...,...,...
2430072,4320484.0,4320484.0,X-25013649-25013697-NGC,chrX,25013649,25013697,3.0,NGC,False,keep
2430073,4376762.0,4376762.0,X-67545316-67545385-GCA,chrX,67545316,67545385,3.0,GCA,False,keep
2430074,4482278.0,4482278.0,X-140504316-140504361-NGC,chrX,140504316,140504361,3.0,NGC,False,keep
2430075,4492755.0,4492755.0,X-147912050-147912110-CGG,chrX,147912050,147912110,3.0,CGG,False,keep


In [9]:
def determine_final_mei_coordinates(row):
    """
    Determine final MEI coordinates based on STR classification rules:
    1. External: Use coordinates of overlapping or closest MEI (upstream if tie)
    2. Non-MEI: Return "non_mei"
    3. Internal fully overlapped: Use MEI coordinates that overlap the STR
    4. Internal between fragments: Combine upstream start with downstream end
    """
    
    # Helper function to parse coordinates
    def parse_coords(coord_str):
        if pd.isna(coord_str) or coord_str == "NA":
            return None, None
        try:
            start, end = coord_str.split(":")
            return int(start), int(end)
        except:
            return None, None
    
    classification = row['STR_classification']
    
    # Case 1: Non-MEI STRs
    if classification == "non_mei":
        return "non_mei", "non_mei", "non_mei", "non_mei"
    
    # Parse upstream and downstream coordinates
    up_start, up_end = parse_coords(row['upstream_coords'])
    down_start, down_end = parse_coords(row['downstream_coords'])
    
    # Case 2: External STRs
    if row['internal_external'] == "external":
        # Priority order: overlapping > proximity, upstream > downstream if tie
        # But avoid NA values when non-NA values are available
        
        # Check for overlapping MEIs first (distance = 0)
        if row['upstream_distance'] == 0 and up_start is not None:  # Upstream overlap
            return up_start, up_end, row['upstream_ID'], row['upstream_family']
        elif row['downstream_distance'] == 0 and down_start is not None:  # Downstream overlap
            return down_start, down_end, row['downstream_ID'], row['downstream_family']
        
        # No overlap, choose closest available MEI
        # Convert distances to numeric, handling NA values
        up_dist = float('inf')
        down_dist = float('inf')
        
        if row['upstream_distance'] != "NA" and up_start is not None:
            try:
                up_dist = float(row['upstream_distance'])
            except:
                up_dist = float('inf')
        
        if row['downstream_distance'] != "NA" and down_start is not None:
            try:
                down_dist = float(row['downstream_distance'])
            except:
                down_dist = float('inf')
        
        # Choose the closest available MEI (upstream if tie)
        if up_dist <= down_dist and up_dist != float('inf'):
            return up_start, up_end, row['upstream_ID'], row['upstream_family']
        elif down_dist != float('inf'):
            return down_start, down_end, row['downstream_ID'], row['downstream_family']
        else:
            return "NA", "NA", "NA", "NA"
    
    # Case 3: Internal STRs
    elif row['internal_external'] == "internal":
        
        # Internal fully overlapped: use the MEI coordinates that overlap
        if classification == "internal_fully_overlapped":
            # Should be same coordinates upstream and downstream
            if up_start is not None:
                return up_start, up_end, row['upstream_ID'], row['upstream_family']
            elif down_start is not None:
                return down_start, down_end, row['downstream_ID'], row['downstream_family']
        
        # Internal between fragments: combine upstream start with downstream end
        elif classification == "internal_between_fragments":
            if up_start is not None and down_end is not None:
                # Use the same MEI ID (should be same for both fragments)
                mei_id = row['upstream_ID'] if row['upstream_ID'] != "NA" else row['downstream_ID']
                mei_family = row['upstream_family'] if row['upstream_family'] != "NA" else row['downstream_family']
                return up_start, down_end, mei_id, mei_family
            elif up_start is not None:  # Fallback to upstream only
                return up_start, up_end, row['upstream_ID'], row['upstream_family']
            elif down_start is not None:  # Fallback to downstream only
                return down_start, down_end, row['downstream_ID'], row['downstream_family']
        
        # Other internal types (internal_overlapped)
        else:
            if up_start is not None:
                return up_start, up_end, row['upstream_ID'], row['upstream_family']
            elif down_start is not None:
                return down_start, down_end, row['downstream_ID'], row['downstream_family']
    
    # Default case
    return "NA", "NA", "NA", "NA"

# Apply the function to determine final MEI coordinates
print("Determining final MEI coordinates...")
coords_result = revised_labels.apply(determine_final_mei_coordinates, axis=1, result_type='expand')
revised_labels[['final_mei_start', 'final_mei_end', 'final_mei_id', 'final_mei_family']] = coords_result

# Display summary
print("\nFinal MEI coordinate assignment summary:")
print("=" * 50)

# Count by classification
# coord_summary = revised_labels.groupby('STR_classification').agg({
#     'final_mei_start': lambda x: sum(x != "NA" and x != "non_mei"),
#     'final_mei_id': lambda x: sum(x != "NA" and x != "non_mei")
# }).rename(columns={'final_mei_start': 'coords_assigned', 'final_mei_id': 'ids_assigned'})

# print(coord_summary)

print(f"\nTotal STRs with MEI coordinates assigned: {sum(revised_labels['final_mei_start'].isin(['NA', 'non_mei']) == False)}")
print(f"Total non-MEI STRs: {sum(revised_labels['final_mei_start'] == 'non_mei')}")
print(f"Total STRs without coordinates: {sum(revised_labels['final_mei_start'] == 'NA')}")

# Show family distribution
print(f"\nMEI family distribution:")
family_counts = revised_labels[revised_labels['final_mei_family'].notna() & 
                              (revised_labels['final_mei_family'] != 'NA') & 
                              (revised_labels['final_mei_family'] != 'non_mei')]['final_mei_family'].value_counts()
print(family_counts)

revised_labels.head()

Determining final MEI coordinates...

Final MEI coordinate assignment summary:

Total STRs with MEI coordinates assigned: 1238616
Total non-MEI STRs: 1191459
Total STRs without coordinates: 0

MEI family distribution:
final_mei_family
SINE/Alu          422749
LINE/L1           402393
LINE/L2           109389
SINE/MIR           98118
LTR/ERVL-MaLR      81719
LTR/ERV1           51480
LTR/ERVL           45893
LINE/CR1            8205
LTR/ERVK            4180
LTR/Gypsy           3096
LINE/RTE-X          2368
Retroposon/SVA      1991
LTR/Gypsy?          1346
LTR                 1317
LINE/RTE-BovB       1039
LTR?                 991
LTR/ERVL?            688
SINE/tRNA-RTE        519
LTR/ERV1?            302
SINE/5S-Deu-L2       286
SINE/tRNA            206
LINE/Dong-R4          87
.                     86
LINE/Penelope         73
LINE/L1-Tx1           40
SINE/tRNA-Deu         37
LINE/Jockey           18
Name: count, dtype: int64


,STR_chr,STR_start,STR_end,STR_ID,upstream_coords,upstream_family,upstream_ID,upstream_distance,downstream_coords,downstream_family,downstream_ID,downstream_distance,STR_classification,internal_external,final_mei_start,final_mei_end,final_mei_id,final_mei_family
0,chr1,14069,14081,15.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
1,chr1,15811,15822,16.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
2,chr1,16619,16632,17.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
3,chr1,16716,16727,18.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
4,chr1,19175,19184,19.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei


# How to move forward with this
We now have new assignments for our final STR set (which includes the updated STRs from VCs and that were discarded incorrectly at the beginning). To move forward with this set we'll have to update the following:
1. We need to use the tab delimited full repeatmasker file to get the subfamilies for the final assigned MEIs as well as orientation (for 3', 5' and internal) as well as divergence
1. After this we can replace the stat_mei and end_mei columns with the final_start_mei and final_end_mei for all downstream analysis
1. Later on, we will have to determine which Alus we changed the start and end coordinates for and rerun our consensus start positions based on the merged fragments (~ 3000 of these)
1. That should be it!

In [10]:
# read the repeatmasker file with standardized headers
# The RepeatMasker file has 15 columns based on the standard .out format
repeatmasker_headers = [
    'SW_score',           # Column 0: Smith-Waterman score
    'perc_div',           # Column 1: % divergence 
    'perc_del',           # Column 2: % deletions
    'perc_ins',           # Column 3: % insertions
    'query_sequence',     # Column 4: query sequence (chromosome)
    'query_begin',        # Column 5: position in query begin
    'query_end',          # Column 6: position in query end
    'query_left',         # Column 7: bases in query sequence left of match
    'strand',             # Column 8: strand (+ or C for complement)
    'matching_repeat',    # Column 9: name of matching repeat
    'repeat_class_family', # Column 10: class/family of repeat
    'repeat_begin',       # Column 11: position in repeat begin
    'repeat_end',         # Column 12: position in repeat end  
    'repeat_left',        # Column 13: bases in repeat left of match
    'ID'                  # Column 14: ID number
]

# Read the file, skipping the first 2 header lines and using our standardized headers
repeatmasker_full = pd.read_csv("/data/projects/nanopore/RepeatExpansion/coordinates/repeatmasker_hg38.fa.out.tab", 
                                sep="\t", 
                                header=None, 
                                skiprows=2,  # Skip the 2-line header
                                names=repeatmasker_headers)

print(f"RepeatMasker file loaded with {len(repeatmasker_full):,} entries")
print("\nColumn names:")
for i, col in enumerate(repeatmasker_headers):
    print(f"  {i:2d}: {col}")

print(f"\nFirst few rows:")
repeatmasker_full.head()

RepeatMasker file loaded with 5,622,516 entries

Column names:
   0: SW_score
   1: perc_div
   2: perc_del
   3: perc_ins
   4: query_sequence
   5: query_begin
   6: query_end
   7: query_left
   8: strand
   9: matching_repeat
  10: repeat_class_family
  11: repeat_begin
  12: repeat_end
  13: repeat_left
  14: ID

First few rows:


,SW_score,perc_div,perc_del,perc_ins,query_sequence,query_begin,query_end,query_left,strand,matching_repeat,repeat_class_family,repeat_begin,repeat_end,repeat_left,ID
0,463,1.3,0.6,1.7,chr1,10001,10468,(248945954),+,(TAACCC)n,Simple_repeat,1,463,(0),1.0
1,4005,11.3,21.5,1.3,chr1,10469,11447,(248944975),C,TAR1,Satellite/telo,(399),1712,483,2.0
2,535,21.2,15.9,3.1,chr1,11485,11676,(248944746),C,L1MC5a,LINE/L1,(510),5667,5447,3.0
3,263,29.4,1.9,1.0,chr1,11678,11780,(248944642),C,MER5B,DNA/hAT-Charlie,(74),104,1,4.0
4,309,23.0,3.7,0.0,chr1,15265,15355,(248941067),C,MIR3,SINE/MIR,(119),143,49,5.0


In [11]:
# get the matching_repeat, strand, perc_div, and ID columns and merge with revised_labels on ID and final_mei_id
mei_rows = revised_labels[revised_labels['final_mei_id'] != 'non_mei']
non_mei_rows = revised_labels[revised_labels['final_mei_id'] == 'non_mei']
mei_rows['final_mei_id'] = mei_rows['final_mei_id'].astype(float)
mei_rows = mei_rows.merge(
    repeatmasker_full[['matching_repeat', 'strand', 'perc_div', 'ID']],
    left_on='final_mei_id',
    right_on='ID',
    how='left',
    suffixes=('', '_rm')
)
# recombine with non_mei rows
revised_labels = pd.concat([mei_rows, non_mei_rows], ignore_index=True)

# Handle rows where final_mei_id is 'non_mee'
revised_labels['matching_repeat'].fillna('non_mei', inplace=True)
revised_labels['strand'].fillna('non_mei', inplace=True)
revised_labels['perc_div'].fillna('non_mei', inplace=True)

revised_labels

/tmp/ipykernel_2341472/3878657949.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  mei_rows['final_mei_id'] = mei_rows['final_mei_id'].astype(float)
/tmp/ipykernel_2341472/3878657949.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  revised_labels['matching_repeat'].fillna('non_mei

,STR_chr,STR_start,STR_end,STR_ID,upstream_coords,upstream_family,upstream_ID,upstream_distance,downstream_coords,downstream_family,...,STR_classification,internal_external,final_mei_start,final_mei_end,final_mei_id,final_mei_family,matching_repeat,strand,perc_div,ID
0,chr1,26932,26941,26.0,26791:27053,SINE/Alu,19.0,0.0,26791:27053,SINE/Alu,...,internal_fully_overlapped,internal,26791,27053,19.0,SINE/Alu,AluSp,+,9.5,19.0
1,chr1,27863,27872,27.0,27833:28014,SINE/MIR,21.0,0.0,27833:28014,SINE/MIR,...,internal_fully_overlapped,internal,27833,28014,21.0,SINE/MIR,MIRb,C,33.6,21.0
2,chr1,28246,28255,28.0,28151:28302,SINE/MIR,22.0,0.0,28151:28302,SINE/MIR,...,internal_fully_overlapped,internal,28151,28302,22.0,SINE/MIR,MIR,C,32.0,22.0
3,chr1,30867,30892,34.0,30694:30848,LTR/ERVL-MaLR,26.0,20.0,NaN,NaN,...,external_upstream_proximity,external,30694,30848,26.0,LTR/ERVL-MaLR,MLT1A,+,18.7,26.0
4,chr1,30867,30892,34.0,30694:30848,LTR/ERVL-MaLR,26.0,20.0,NaN,NaN,...,external_upstream_proximity,external,30694,30848,26.0,LTR/ERVL-MaLR,MLT1A,+,24.9,26.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3375485,chrY,56694118,56694136,4542235.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375486,chrY,56694221,56694236,4542236.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375487,chrY,56694333,56694350,4542237.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375488,chrY,56694379,56694395,4542238.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN


In [12]:
# where were the duplicates introduced?
# print the duplicated rows that are not complete duplicates
revised_labels.drop_duplicates(subset=["STR_ID", "final_mei_id"], keep="first", inplace=True)
revised_labels

,STR_chr,STR_start,STR_end,STR_ID,upstream_coords,upstream_family,upstream_ID,upstream_distance,downstream_coords,downstream_family,...,STR_classification,internal_external,final_mei_start,final_mei_end,final_mei_id,final_mei_family,matching_repeat,strand,perc_div,ID
0,chr1,26932,26941,26.0,26791:27053,SINE/Alu,19.0,0.0,26791:27053,SINE/Alu,...,internal_fully_overlapped,internal,26791,27053,19.0,SINE/Alu,AluSp,+,9.5,19.0
1,chr1,27863,27872,27.0,27833:28014,SINE/MIR,21.0,0.0,27833:28014,SINE/MIR,...,internal_fully_overlapped,internal,27833,28014,21.0,SINE/MIR,MIRb,C,33.6,21.0
2,chr1,28246,28255,28.0,28151:28302,SINE/MIR,22.0,0.0,28151:28302,SINE/MIR,...,internal_fully_overlapped,internal,28151,28302,22.0,SINE/MIR,MIR,C,32.0,22.0
3,chr1,30867,30892,34.0,30694:30848,LTR/ERVL-MaLR,26.0,20.0,NaN,NaN,...,external_upstream_proximity,external,30694,30848,26.0,LTR/ERVL-MaLR,MLT1A,+,18.7,26.0
5,chr1,30935,30948,36.0,NaN,NaN,NaN,NaN,30953:31131,LTR/ERVL-MaLR,...,external_downstream_proximity,external,30953,31131,26.0,LTR/ERVL-MaLR,MLT1A,+,18.7,26.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3375485,chrY,56694118,56694136,4542235.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375486,chrY,56694221,56694236,4542236.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375487,chrY,56694333,56694350,4542237.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN
3375488,chrY,56694379,56694395,4542238.0,NaN,NaN,NaN,NaN,NaN,NaN,...,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,NaN


In [13]:
final_labels = revised_labels[["STR_chr","STR_start","STR_end","STR_ID","final_mei_id","final_mei_start","final_mei_end","final_mei_family","matching_repeat","strand","perc_div","STR_classification","internal_external"]]
final_labels.columns = ["chr","start","end","LocusID","mei_id","start_mei","end_mei","mei_family","mei_subfamily","orientation","divergence","STR_classification","internal_external"]
final_labels

,chr,start,end,LocusID,mei_id,start_mei,end_mei,mei_family,mei_subfamily,orientation,divergence,STR_classification,internal_external
0,chr1,26932,26941,26.0,19.0,26791,27053,SINE/Alu,AluSp,+,9.5,internal_fully_overlapped,internal
1,chr1,27863,27872,27.0,21.0,27833,28014,SINE/MIR,MIRb,C,33.6,internal_fully_overlapped,internal
2,chr1,28246,28255,28.0,22.0,28151,28302,SINE/MIR,MIR,C,32.0,internal_fully_overlapped,internal
3,chr1,30867,30892,34.0,26.0,30694,30848,LTR/ERVL-MaLR,MLT1A,+,18.7,external_upstream_proximity,external
5,chr1,30935,30948,36.0,26.0,30953,31131,LTR/ERVL-MaLR,MLT1A,+,18.7,external_downstream_proximity,external
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3375485,chrY,56694118,56694136,4542235.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
3375486,chrY,56694221,56694236,4542236.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
3375487,chrY,56694333,56694350,4542237.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei
3375488,chrY,56694379,56694395,4542238.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei


In [14]:
final_labels = final_labels.merge(updated_labels[["LocusID", "original_LocusID","TRID","str_period","str_motif"]], on="LocusID", suffixes=("", "_orig"), how = "left").drop_duplicates(subset=["LocusID"])
final_labels

,chr,start,end,LocusID,mei_id,start_mei,end_mei,mei_family,mei_subfamily,orientation,divergence,STR_classification,internal_external,original_LocusID,TRID,str_period,str_motif
0,chr1,26932,26941,26.0,19.0,26791,27053,SINE/Alu,AluSp,+,9.5,internal_fully_overlapped,internal,26.0,1-26932-26941-GTG,3.0,GTG
1,chr1,27863,27872,27.0,21.0,27833,28014,SINE/MIR,MIRb,C,33.6,internal_fully_overlapped,internal,27.0,1-27863-27872-ATC,3.0,ATC
2,chr1,28246,28255,28.0,22.0,28151,28302,SINE/MIR,MIR,C,32.0,internal_fully_overlapped,internal,28.0,1-28246-28255-GTA,3.0,GTA
3,chr1,30867,30892,34.0,26.0,30694,30848,LTR/ERVL-MaLR,MLT1A,+,18.7,external_upstream_proximity,external,34.0,1-30867-30892-CT,2.0,CT
4,chr1,30935,30948,36.0,26.0,30953,31131,LTR/ERVL-MaLR,MLT1A,+,18.7,external_downstream_proximity,external,36.0,1-30935-30948-TC,2.0,TC
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2430070,chrY,56694118,56694136,4542235.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,4542235.0,Y-56694118-56694136-AATGG,5.0,AATGG
2430071,chrY,56694221,56694236,4542236.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,4542236.0,Y-56694221-56694236-GGAAT,5.0,GGAAT
2430072,chrY,56694333,56694350,4542237.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,4542237.0,Y-56694333-56694350-AATGG,5.0,AATGG
2430073,chrY,56694379,56694395,4542238.0,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,non_mei,4542238.0,Y-56694379-56694395-ATGGA,5.0,ATGGA


# Add additional labels
1. TE (mei) labels
1. finalize terminal labels
1. genomic annotations

In [15]:
final_labels["mei"] = final_labels["internal_external"] != "non_mei"
# update relative position labels
# Cast only numeric rows of start_mei and end_mei to int, leaving non-numeric rows unchanged
final_labels["start_mei"] = pd.to_numeric(final_labels["start_mei"], errors='coerce').astype('Int64')
final_labels["end_mei"] = pd.to_numeric(final_labels["end_mei"], errors='coerce').astype('Int64')

final_labels.loc[final_labels["mei"], "terminal_label"] = final_labels.loc[final_labels["mei"]].apply(
    SequenceAttributes.get_terminal_label, axis=1, args=(0, 10)
)
final_labels["terminal_label"] = final_labels["terminal_label"].fillna("non_mei")


final_labels.loc[final_labels["mei_family"] == "Retroposon/SVA", "terminal_label"] = final_labels.loc[final_labels["mei_family"] == "Retroposon/SVA"].apply(SequenceAttributes.get_terminal_label, axis=1, args=(20, 10)
)
final_labels.loc[final_labels["mei_family"] == "Retroposon/SVA", "terminal_label"]


1998         internal
1999       5'terminal
2218         internal
2219       5'terminal
2948       5'terminal
              ...    
1232628    5'terminal
1232650    5'terminal
1235907    5'terminal
1235908      internal
1235909      internal
Name: terminal_label, Length: 1989, dtype: object

In [16]:
genomic_annotations = pd.read_csv("/nfs/boylelab_turbo/kvandeyn/MEI_STR/data/TR_catalog_HPRC/updated_labels_TRID_gene_annotations.csv")
print(genomic_annotations.head())
final_labels = final_labels.merge(genomic_annotations, on="LocusID", how="left")
final_labels.GencodeGeneRegion.value_counts()

   LocusID GencodeGeneRegion RefseqGeneRegion ManeGeneRegion
0     26.0        intergenic           intron     intergenic
1     37.0        intergenic           intron     intergenic
2     38.0        intergenic           intron     intergenic
3     46.0            intron           intron     intergenic
4     51.0        intergenic       intergenic     intergenic


GencodeGeneRegion
intron        1321775
intergenic     964609
CDS             36466
exon            35587
3' UTR          29940
promoter        29314
5' UTR          12352
Name: count, dtype: int64

In [17]:
# update the disease associated regions that we remapped the locusids for
daTR_mappings = {3240174.0:"intron",922520.0:"intron",4542951.0:"CDS",4053467.0:"intron",1033259.0:"CDS",4542947.0:"CDS",4542948.0:"CDS",4542949.0:"CDS",4542950.0:"CDS",
4542945.0:"intron",
4542946.0:"intron",
4542953.0: "intron",
3192251.0: "CDS"
}
# add the remapped disease associated regions to the final_labels dataframe
for locusid, region in daTR_mappings.items():
    final_labels.loc[final_labels["LocusID"] == locusid, "GencodeGeneRegion"] = region
final_labels["genomic_annotation"] = final_labels["GencodeGeneRegion"]
final_labels.GencodeGeneRegion.value_counts()

GencodeGeneRegion
intron        1321781
intergenic     964609
CDS             36473
exon            35587
3' UTR          29940
promoter        29314
5' UTR          12352
Name: count, dtype: int64

In [18]:
# drop duplicate rows by locusID
final_labels = final_labels.drop_duplicates(subset="LocusID")
# label regions that are ambiguous interms of 5' and 3' labeling
final_labels["keep_rel_pos"] = ~((final_labels.terminal_label == "5'terminal") & (final_labels.STR_classification == "external_different_meis"))
final_labels.keep_rel_pos.value_counts()

keep_rel_pos
True     2410619
False      19437
Name: count, dtype: int64

In [21]:
# trial out adding disease association to this data
daTR_coords = pd.read_csv("daTR_coords.bed",sep="\t")
# name the columns such that they are compatible with pyranges
daTR_coords.columns = ["Chromosome","Start","End","motif","disease_id"]
daTR_ranges = pr.PyRanges(daTR_coords[['Chromosome','Start','End','disease_id']])

final_labels_pyranges = pr.PyRanges(final_labels[['chr','start','end']].drop_duplicates().rename(columns={"chr":"Chromosome","start":"Start","end":"End"}))

# find overlaps
overlaps = final_labels_pyranges.join(daTR_ranges)
# merge back to the data
temp = pd.merge(left = final_labels, right = overlaps.df.drop_duplicates(subset="disease_id"), left_on = ["chr","start","end"],right_on = ["Chromosome","Start","End"], how = "left")
final_labels = pd.merge(left = final_labels, right = temp[["TRID","disease_id"]], on = "TRID", how = "left").drop_duplicates()
final_labels["disease_id"] = final_labels["disease_id"].fillna("non_disease_associated")
final_labels["disease_associated"] = final_labels["disease_id"] != "non_disease_associated"
final_labels["disease_id"].value_counts()

disease_id
non_disease_associated    2429992
HD                              5
DM2                             3
SCA36                           2
SCA8                            2
SCA31                           1
FAME2                           1
FAME6                           1
SCA37                           1
FAME4                           1
CANVAS                          1
FAME1                           1
FAME3                           1
DMD                             1
XDP                             1
NIID                            1
FA                              1
OPML1                           1
FRA11B                          1
FRA12A                          1
DRPLA                           1
SCA2                            1
HPE                             1
SCA10                           1
SCA27B                          1
SCA3                            1
OPMD                            1
HDL2                            1
ALS                             1
SCA

In [22]:
final_labels["disease_associated"].value_counts()

disease_associated
False    2429992
True          64
Name: count, dtype: int64

In [23]:
# separate the disease associated vs non-disease associated spans
hd_mask = (final_labels["disease_id"] == "HD") & final_labels["disease_associated"]
not_hd_locus = hd_mask & (final_labels["LocusID"] != 1033259.0)

final_labels.loc[not_hd_locus, "disease_id"] = "non_disease_associated"
final_labels.loc[not_hd_locus, "disease_associated"] = False

#DM2: 4542945.0 is the disease associated locus for DM2
dm2_mask = (final_labels["disease_id"] == "DM2") & final_labels["disease_associated"] & (final_labels["LocusID"] != 4542945.0)
final_labels.loc[dm2_mask, "disease_id"] = "non_disease_associated"
final_labels.loc[dm2_mask, "disease_associated"] = False

# SCA36 locus should only be LocusID 4053467.0
sca36_mask = (final_labels["disease_id"] == "SCA36") & final_labels["disease_associated"] & (final_labels["LocusID"] != 4053467.0)
final_labels.loc[sca36_mask, "disease_id"] = "non_disease_associated"
final_labels.loc[sca36_mask, "disease_associated"] = False

# SCA8 locus should only be LocusID 4542951.0
sca8_mask = (final_labels["disease_id"] == "SCA8") & final_labels["disease_associated"] & (final_labels["LocusID"] != 4542951.0)
final_labels.loc[sca8_mask, "disease_id"] = "non_disease_associated"
final_labels.loc[sca8_mask, "disease_associated"] = False

# change the DM2 row to have str_motif CAGG and the SCA27B row to GAA
final_labels.loc[final_labels.disease_id == "DM2", "str_motif"] = "CAGG"
final_labels.loc[final_labels.disease_id == "SCA27B", "str_motif"] = "GAA"

final_labels[final_labels.disease_associated]

,chr,start,end,LocusID,mei_id,start_mei,end_mei,mei_family,mei_subfamily,orientation,...,str_motif,mei,terminal_label,GencodeGeneRegion,RefseqGeneRegion,ManeGeneRegion,genomic_annotation,keep_rel_pos,disease_id,disease_associated
28715,chr1,57367043,57367118,101068.0,112245.0,57366925,57367291,SINE/Alu,AluJb,+,...,AAAAT,True,internal,intron,intron,intron,intron,True,SCA37,True
306137,chr13,102161564,102161844,3240174.0,1132019.0,102161732,102161835,LINE/L2,L2c,+,...,GAA,True,5'terminal,intron,NaN,NaN,intron,True,SCA27B,True
395308,chr16,24613439,24613529,3573818.0,1472389.0,24613538,24613835,SINE/Alu,AluSx1,C,...,TTTTA,True,3'terminal,intergenic,intron,intergenic,intergenic,True,FAME6,True
410018,chr16,66490395,66490467,3631877.0,1527859.0,66490100,66490396,SINE/Alu,AluSx,+,...,TAAAA,True,3'terminal,intron,intron,intergenic,intron,True,SCA31,True
556480,chr2,96197066,96197121,511206.0,2088546.0,96196786,96197066,SINE/Alu,AluSp,+,...,AAAAT,True,3'terminal,intron,intron,intron,intron,True,FAME2,True
673034,chr22,45795354,45795424,4270367.0,2528779.0,45795425,45795572,SINE/Alu,AluSx,C,...,ATTCT,True,3'terminal,intron,intron,intron,intron,True,SCA10,True
726227,chr3,129172576,129172732,4542945.0,2731440.0,129172300,129172573,SINE/Alu,AluSx,+,...,CAGG,True,3'terminal,intron,NaN,NaN,intron,True,DM2,True
749279,chr3,183712187,183712222,1004330.0,2813123.0,183712227,183712510,SINE/Alu,AluSx,C,...,TTTTA,True,3'terminal,intron,intron,intron,intron,True,FAME4,True
771976,chr4,39348424,39348479,1087753.0,2900981.0,39348146,39348424,SINE/Alu,AluSx3,+,...,AAAAG,True,3'terminal,intron,intron,intron,intron,True,CANVAS,True
836926,chr5,10356346,10356411,1324732.0,3122378.0,10356414,10356695,SINE/Alu,AluSx,C,...,ATTTT,True,3'terminal,intron,intron,intron,intron,True,FAME3,True


In [24]:
# lastly, add the simple_motif
final_labels = final_labels[~final_labels.str_motif.str.contains(",")]
final_labels["simple_motif"] = final_labels["str_motif"].apply(SequenceAttributes.get_simple_motif)

In [31]:
# save the final_labels
# replace / in mei_family with _
final_labels.mei_family = final_labels.mei_family.str.replace("/", "_")
final_labels.to_csv("final_manuscript_labels.tsv",sep="\t", index=False)